# Эксперимент 3: Adam, масштаб признаков и редкие аномалии

Первая часть проверяет автоматическое диагональное масштабирование Adam на данных с искусственно разными масштабами признаков. Вторая часть воспроизводит скачки Adam на редких тяжелых аномалиях.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
from src.experiments import make_anomaly_oracle, make_scaled_oracle, make_team_oracles, plot_histories
from src.optimization import adam, sgd

FIGS = ROOT / 'figs'
FIGS.mkdir(exist_ok=True)

In [ ]:
base_oracle = make_team_oracles(m=2000, n=30, regcoef=1e-2, random_state=11)['Log-Cosh']
scaled_oracle = make_scaled_oracle(base_oracle)
x0 = np.zeros(scaled_oracle.n)

histories = {}
_, _, histories['SGD tuned'] = sgd(
    scaled_oracle, x0, batch_size=32, max_epoch=40,
    lr_schedule='constant', lr_params={'alpha_0': 1e-8}, trace=True, random_state=3
)
_, _, histories['Adam alpha=0.001'] = adam(
    scaled_oracle, x0, step_size=1e-3, batch_size=32, max_epoch=40, trace=True, random_state=3
)
plot_histories(histories, y_key='func', title='Масштабированные признаки: SGD vs Adam')
plt.savefig(FIGS / '03_adam_scaled_features.png', dpi=200, bbox_inches='tight')

In [ ]:
anom_oracle, anomaly_idx = make_anomaly_oracle(base_oracle, fraction=0.01, multiplier=100.0, random_state=12)
print(f'Аномалий: {len(anomaly_idx)} из {anom_oracle.m}')

histories = {}
_, _, histories['SGD constant'] = sgd(
    anom_oracle, np.zeros(anom_oracle.n), batch_size=5, max_epoch=120,
    lr_schedule='constant', lr_params={'alpha_0': 1e-4}, trace=True, random_state=4
)
_, _, histories['Adam beta2=0.99'] = adam(
    anom_oracle, np.zeros(anom_oracle.n), step_size=1e-3, batch_size=5,
    max_epoch=120, beta2=0.99, trace=True, random_state=4
)
plot_histories(histories, y_key='func', title='Редкие тяжелые аномалии: скачки функции')
plt.savefig(FIGS / '03_adam_heavy_anomalies.png', dpi=200, bbox_inches='tight')

В первой части знаменатель `sqrt(v_hat)` работает как диагональный предобуславливатель. Во второй части редкие большие градиенты приходят после того, как Adam успел забыть старые большие значения в `v_k`, поэтому эффективный шаг может стать аномально большим.